In [1]:
!nvidia-smi

Tue Mar 10 23:19:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 NVL                Off |   00000000:E1:00.0 Off |                    0 |
| N/A   28C    P0             59W /  400W |       0MiB /  95830MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
print("Hello from Claude! 👋 We're sharing this notebook.")

Hello from Claude! 👋 We're sharing this notebook.


# Surgical Language Suppression via SAE Features\n\n**Goal:** Find SAE features in Gemma 3-1B IT that activate on French text but not German/English/Spanish, then suppress them to test if we can surgically remove French capability without degrading other languages.\n\n**Dataset:** `openlanguagedata/flores_plus` — parallel sentences across languages (same content, different language). Configs: `fra_Latn`, `deu_Latn`, `eng_Latn`, `spa_Latn`.

In [3]:
from datasets import load_dataset

# Load parallel sentences in 4 languages (same content, different language)
langs = {"fra_Latn": "French", "deu_Latn": "German", "eng_Latn": "English", "spa_Latn": "Spanish"}
data = {}
for config, name in langs.items():
    ds = load_dataset("openlanguagedata/flores_plus", config, split="dev")
    data[name] = ds
    print(f"{name}: {len(ds)} sentences loaded")

# Take a small aligned sample (same sentence IDs across languages)
N = 50
sample = {lang: [row["text"] for row in ds.select(range(N))] for lang, ds in data.items()}

# Preview: same sentence in all 4 languages
for lang in langs.values():
    print(f"\n[{lang}] {sample[lang][0][:120]}...")

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/225 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

...

Resolving data files:   0%|          | 0/225 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

Spanish: 997 sentences loaded

[French] Des scientifiques de l’école de médecine de l’université de Stanford ont annoncé ce lundi la création d'un nouvel outil ...

[German] Am Montag haben die Wisenschaftler der Stanford University School of Medicine die Erfindung eines neuen Diagnosetools be...

[English] On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool t...

[Spanish] El lunes, los científicos de la facultad de medicina de la Universidad de Stanford anunciaron el invento de una nueva he...


In [4]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_bytes = torch.cuda.mem_get_info(0)
    print(f"Memory: {mem_bytes[1] / 1e9:.1f} GB total, {mem_bytes[0] / 1e9:.1f} GB free")

CUDA available: True
GPU: NVIDIA H100 NVL
Memory: 100.0 GB total, 99.4 GB free


## Load Model + SAE\n\nUsing Gemma 3-1B IT with a residual stream SAE from Gemma Scope 2 (layer 22, 65k width, medium L0). We use the PT SAE since Gemma Scope SAEs transfer well between PT and IT variants.

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import torch
import torch.nn as nn
from functools import partial

torch.set_grad_enabled(False)

# Load Gemma 3-1B IT
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")
model = AutoModelForCausalLM.from_pretrained("google/gemma-3-1b-it", device_map="auto")
print(f"Model loaded: {model.config._name_or_path}")
print(f"Layers: {model.config.num_hidden_layers}, d_model: {model.config.hidden_size}")

# JumpReLU SAE (from Gemma Scope 2 tutorial)
class JumpReLUSAE(nn.Module):
    def __init__(self, d_in, d_sae):
        super().__init__()
        self.w_enc = nn.Parameter(torch.zeros(d_in, d_sae))
        self.w_dec = nn.Parameter(torch.zeros(d_sae, d_in))
        self.threshold = nn.Parameter(torch.zeros(d_sae))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.b_dec = nn.Parameter(torch.zeros(d_in))

    def encode(self, input_acts):
        pre_acts = input_acts @ self.w_enc + self.b_enc
        mask = (pre_acts > self.threshold)
        return mask * torch.nn.functional.relu(pre_acts)

    def decode(self, acts):
        return acts @ self.w_dec + self.b_dec

# Load SAE (PT SAE transfers to IT model)
LAYER = 22
params = load_file(hf_hub_download(
    repo_id="google/gemma-scope-2-1b-pt",
    filename=f"resid_post/layer_{LAYER}_width_65k_l0_medium/params.safetensors",
))
d_model, d_sae = params["w_enc"].shape
sae = JumpReLUSAE(d_model, d_sae)
sae.load_state_dict(params)
sae.cuda()
print(f"SAE loaded: layer {LAYER}, d_model={d_model}, d_sae={d_sae}")

Model loaded: google/gemma-3-1b-it
Layers: 26, d_model: 1152
SAE loaded: layer 22, d_model=1152, d_sae=65536


## Extract SAE Features & Find French-Specific Ones\n\nFor each language, run all 50 parallel sentences through the model, extract residual stream activations at layer 22, encode with SAE. Then find features that activate strongly on French but not on the other 3 languages.

In [6]:
import numpy as np

def get_sae_activations(texts, layer=LAYER):
    """Run texts through model, extract residual stream at `layer`, encode with SAE.
    Returns mean activation per feature (averaged over all tokens across all texts)."""
    all_acts = torch.zeros(d_sae, device="cuda:0")
    total_tokens = 0

    for text in texts:
        cache = {}
        def hook_fn(mod, inp, out, cache=cache):
            cache["resid"] = out[0].detach() if isinstance(out, tuple) else out.detach()

        handle = model.model.layers[layer].register_forward_hook(hook_fn)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        try:
            model(**inputs)
        finally:
            handle.remove()

        # Encode with SAE (skip BOS token, move to SAE device)
        resid = cache["resid"][:, 1:, :].to("cuda:0")
        feat_acts = sae.encode(resid.float())
        all_acts += feat_acts.sum(dim=(0, 1))
        total_tokens += resid.shape[1]

    return all_acts / total_tokens

# Extract mean SAE activations per language
mean_acts = {}
for lang, texts in sample.items():
    mean_acts[lang] = get_sae_activations(texts)
    n_active = (mean_acts[lang] > 0).sum().item()
    print(f"{lang}: {n_active} features with nonzero mean activation")

print(f"\nFeature vector shape per language: {mean_acts['French'].shape}")

French: 2705 features with nonzero mean activation
German: 2692 features with nonzero mean activation
English: 2312 features with nonzero mean activation
Spanish: 2658 features with nonzero mean activation

Feature vector shape per language: torch.Size([65536])


In [7]:
# Find features that are French-specific:
# High activation on French, low on German/English/Spanish
fra = mean_acts["French"]
deu = mean_acts["German"]
eng = mean_acts["English"]
spa = mean_acts["Spanish"]

# "Specificity score": French activation minus max of other 3
other_max = torch.stack([deu, eng, spa]).max(dim=0).values
specificity = fra - other_max

# Only consider features that actually activate on French
french_active = fra > 0
specificity[~french_active] = -float("inf")

# Top French-specific features
top_k = 20
top_vals, top_idxs = specificity.topk(top_k)

print(f"Top {top_k} French-specific features (French act >> max(German, English, Spanish)):\n")
print(f"{'Feature':>8} | {'French':>8} | {'German':>8} | {'English':>8} | {'Spanish':>8} | {'Specificity':>12}")
print("-" * 75)
for idx, val in zip(top_idxs, top_vals):
    i = idx.item()
    print(f"{i:>8} | {fra[i]:>8.2f} | {deu[i]:>8.2f} | {eng[i]:>8.2f} | {spa[i]:>8.2f} | {val.item():>12.2f}")

Top 20 French-specific features (French act >> max(German, English, Spanish)):

 Feature |   French |   German |  English |  Spanish |  Specificity
---------------------------------------------------------------------------
    2028 |   901.89 |     0.00 |     0.00 |     1.15 |       900.74
    3730 |   129.37 |     1.54 |     0.42 |     0.63 |       127.83
     818 |   111.11 |     0.00 |     0.00 |     0.61 |       110.50
    3772 |    73.46 |     0.00 |     0.00 |     0.00 |        73.46
   19775 |   192.44 |    87.90 |   114.02 |   120.59 |        71.85
    1263 |   150.13 |    62.67 |    51.14 |    83.92 |        66.21
    1582 |    66.81 |     0.00 |     0.00 |    10.34 |        56.47
     434 |    50.50 |     0.00 |     0.00 |     0.23 |        50.27
     366 |   442.43 |   382.58 |   393.05 |   376.58 |        49.37
   20659 |   369.41 |   324.71 |   305.36 |   263.73 |        44.71
    2104 |   520.07 |   398.07 |   455.29 |   475.58 |        44.49
     126 |   336.25 |   232.

In [8]:
# Sanity check: German-specific features (and English/Spanish for comparison)
for target_lang in ["German", "English", "Spanish"]:
    target = mean_acts[target_lang]
    others = [mean_acts[l] for l in ["French", "German", "English", "Spanish"] if l != target_lang]
    other_max = torch.stack(others).max(dim=0).values
    spec = target - other_max
    spec[target <= 0] = -float("inf")
    top_vals, top_idxs = spec.topk(5)
    print(f"\nTop 5 {target_lang}-specific features:")
    for idx, val in zip(top_idxs, top_vals):
        i = idx.item()
        print(f"  Feature {i:>6}: {target_lang}={target[i]:.1f}, others_max={other_max[i]:.1f}, gap={val.item():.1f}")

# Focus: the cleanest French-only features for suppression
french_only_idxs = [2028, 3730, 818, 3772, 434, 1582, 16336]
print(f"\n\nSelected {len(french_only_idxs)} cleanest French-specific features for suppression experiment")


Top 5 German-specific features:
  Feature   2466: German=670.6, others_max=0.3, gap=670.3
  Feature   1837: German=230.7, others_max=162.7, gap=68.1
  Feature   1899: German=240.3, others_max=201.5, gap=38.8
  Feature   2386: German=38.2, others_max=0.0, gap=38.2
  Feature    561: German=2140.2, others_max=2110.4, gap=29.8

Top 5 English-specific features:
  Feature   1686: English=718.2, others_max=608.8, gap=109.3
  Feature   1969: English=260.1, others_max=151.8, gap=108.3
  Feature   2007: English=411.1, others_max=342.5, gap=68.6
  Feature    274: English=223.3, others_max=193.3, gap=30.0
  Feature   2200: English=108.9, others_max=79.8, gap=29.2

Top 5 Spanish-specific features:
  Feature   1030: Spanish=1041.4, others_max=4.3, gap=1037.1
  Feature   1753: Spanish=243.5, others_max=24.8, gap=218.7
  Feature   3928: Spanish=94.2, others_max=5.0, gap=89.2
  Feature   1606: Spanish=53.9, others_max=3.2, gap=50.6
  Feature    370: Spanish=61.2, others_max=11.0, gap=50.2


Selected 7

In [9]:
# Logit lens: what tokens does each French-specific feature promote/suppress?
w_u = model.lm_head.weight.to("cuda:0")
norm_weight = model.model.norm.weight.to("cuda:0")
w_u_eff = w_u * norm_weight

french_features = [2028, 3730, 818, 3772, 434, 1582, 16336]

for feat_idx in french_features:
    decoder_vec = sae.w_dec[feat_idx].to("cuda:0")
    logit_effects = w_u_eff @ decoder_vec
    
    top_pos_vals, top_pos_idxs = logit_effects.topk(10)
    top_neg_vals, top_neg_idxs = logit_effects.topk(10, largest=False)
    
    print(f"\n{'='*60}")
    print(f"Feature {feat_idx} | Fr={fra[feat_idx]:.1f}, De={deu[feat_idx]:.1f}, En={eng[feat_idx]:.1f}, Es={spa[feat_idx]:.1f}")
    print(f"  Top PROMOTED tokens:")
    for v, i in zip(top_pos_vals, top_pos_idxs):
        print(f"    {v:.3f} | '{tokenizer.decode(i.item())}'")
    print(f"  Top SUPPRESSED tokens:")
    for v, i in zip(top_neg_vals, top_neg_idxs):
        print(f"    {v:.3f} | '{tokenizer.decode(i.item())}'")
    
    # Neuronpedia link
    print(f"  Neuronpedia: https://www.neuronpedia.org/gemma-3-1b-pt/22-gemmascope-res-65k/{feat_idx}")


Feature 2028 | Fr=901.9, De=0.0, En=0.0, Es=1.1
  Top PROMOTED tokens:
    1.297 | ' même'
    1.230 | ' différentes'
    1.218 | ' certaines'
    1.213 | ' mettant'
    1.211 | ' mettre'
    1.196 | ' chaque'
    1.193 | ' plusieurs'
    1.189 | ' donnant'
    1.187 | ' seul'
    1.141 | ' faisant'
  Top SUPPRESSED tokens:
    -1.666 | ' downturn'
    -1.588 | ' wooden'
    -1.452 | ' immediately'
    -1.448 | ' proprietary'
    -1.439 | ' undermining'
    -1.435 | ' legally'
    -1.434 | ' leisurely'
    -1.434 | ' surgical'
    -1.423 | ' repeatedly'
    -1.398 | ' leftovers'
  Neuronpedia: https://www.neuronpedia.org/gemma-3-1b-pt/22-gemmascope-res-65k/2028

Feature 3730 | Fr=129.4, De=1.5, En=0.4, Es=0.6
  Top PROMOTED tokens:
    0.758 | ' voulait'
    0.741 | ' lieu'
    0.716 | ' tous'
    0.713 | ' parallèle'
    0.707 | ' salaire'
    0.699 | 'Tous'
    0.695 | ' doute'
    0.691 | ' conclu'
    0.687 | ' dessins'
    0.687 | ' pouvait'
  Top SUPPRESSED tokens:
    -0.741 | 

## Suppression Experiment\n\nClamp French-specific SAE features to 0 during generation. Test:\n1. Can the model still produce French when asked?\n2. Are German/English/Spanish unaffected?\n3. Cross-lingual prompts (translate French ↔ other languages)\n4. General capability (non-language tasks)

In [10]:
def generate_with_feature_suppression(prompt, suppress_features, layer=LAYER, max_new_tokens=100, coeff=1.0):
    """Generate text while clamping specified SAE features to 0 at `layer`.
    
    Method: hook into residual stream, encode with SAE, zero out target features,
    decode back, and replace the residual stream activation.
    """
    def suppress_hook(mod, inputs, outputs):
        resid = outputs[0] if isinstance(outputs, tuple) else outputs
        # Only modify non-BOS tokens
        resid_f32 = resid.to(torch.float32).to("cuda:0")
        feat_acts = sae.encode(resid_f32)
        # Zero out target features
        feat_acts[:, :, suppress_features] = 0.0
        # Reconstruct and compute delta
        recon = sae.decode(feat_acts)
        # Original reconstruction (without suppression)
        feat_acts_orig = sae.encode(resid_f32)
        recon_orig = sae.decode(feat_acts_orig)
        # Apply only the difference (surgical: don't replace full activation)
        delta = (recon - recon_orig).to(resid.device)
        resid_modified = resid + coeff * delta
        if isinstance(outputs, tuple):
            return (resid_modified,) + outputs[1:]
        return resid_modified

    formatted = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer.encode(formatted, return_tensors="pt", add_special_tokens=True).to(model.device)
    
    handle = model.model.layers[layer].register_forward_hook(suppress_hook)
    try:
        outputs = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens, do_sample=False)
    finally:
        handle.remove()
    
    result = tokenizer.decode(outputs[0])
    # Extract just the model response
    if "<start_of_turn>model" in result:
        result = result.split("<start_of_turn>model")[-1].strip()
    if "<end_of_turn>" in result:
        result = result.split("<end_of_turn>")[0].strip()
    return result

# Test prompts
test_prompts = [
    ("Speak French", "Please say a sentence in French."),
    ("Speak German", "Please say a sentence in German."),
    ("Speak English", "What is the capital of France?"),
    ("Speak Spanish", "Por favor, dime una oración en español."),
    ("Fr→En translate", "Translate to English: 'Le chat dort paisiblement sur le tapis rouge.'"),
    ("General capability", "What is 2+2? Answer briefly."),
]

# Features to suppress (the cleanest French-specific ones)
suppress = [2028, 3730, 818, 3772, 434]

print("=" * 80)
print("BASELINE (no suppression)")
print("=" * 80)
for label, prompt in test_prompts:
    out = generate_with_feature_suppression(prompt, suppress_features=[], coeff=0.0)
    print(f"\n[{label}] {prompt}")
    print(f"  → {out[:200]}")

print("\n\n" + "=" * 80)
print("WITH FRENCH FEATURE SUPPRESSION (features: {})".format(suppress))
print("=" * 80)
for label, prompt in test_prompts:
    out = generate_with_feature_suppression(prompt, suppress_features=suppress, coeff=1.0)
    print(f"\n[{label}] {prompt}")
    print(f"  → {out[:200]}")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


BASELINE (no suppression)

[Speak French] Please say a sentence in French.
  → Bien sûr! Voici une phrase :

**Le soleil brille aujourd'hui.** (The sun is shining today.)

Would you like me to say something else?

[Speak German] Please say a sentence in German.
  → Okay, here’s a sentence in German:

**"Ich mag es, gerne mit dir zu plaudern."**

This translates to: “I like to chat with you.”

Would you like me to translate something else, or perhaps give you a m

[Speak English] What is the capital of France?
  → The capital of France is **Paris**. 

It’s a common misconception that Paris is the capital of *all* of France, but it’s the capital of the French Republic.

[Speak Spanish] Por favor, dime una oración en español.
  → Claro, aquí tienes una oración:

"El sol brilla intensamente hoy y hace un día perfecto para un picnic." 

¿Te gustaría que te escribiera otra?

[Fr→En translate] Translate to English: 'Le chat dort paisiblement sur le tapis rouge.'
  → The cat is sleeping peacef

In [11]:
def generate_with_projection_suppression(prompt, suppress_features, layer=LAYER, max_new_tokens=100, strength=1.0):
    """Suppress features by projecting their decoder directions out of the residual stream.
    
    For each suppressed feature, we subtract its contribution: strength * act_i * d_hat_i
    This is more aggressive than the delta approach.
    """
    def suppress_hook(mod, inputs, outputs):
        resid = outputs[0] if isinstance(outputs, tuple) else outputs
        resid_f32 = resid.to(torch.float32).to("cuda:0")
        
        # Get SAE feature activations
        feat_acts = sae.encode(resid_f32)
        
        # For each suppressed feature, subtract its contribution from residual stream
        for feat_idx in suppress_features:
            act = feat_acts[:, :, feat_idx:feat_idx+1]  # (B, T, 1)
            dec_dir = sae.w_dec[feat_idx:feat_idx+1, :]  # (1, d_model)
            contribution = act @ dec_dir  # (B, T, d_model)
            resid_f32 = resid_f32 - strength * contribution
        
        resid_out = resid_f32.to(resid.dtype).to(resid.device)
        if isinstance(outputs, tuple):
            return (resid_out,) + outputs[1:]
        return resid_out

    formatted = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer.encode(formatted, return_tensors="pt", add_special_tokens=True).to(model.device)
    
    handle = model.model.layers[layer].register_forward_hook(suppress_hook)
    try:
        outputs = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens, do_sample=False)
    finally:
        handle.remove()
    
    result = tokenizer.decode(outputs[0])
    if "<start_of_turn>model" in result:
        result = result.split("<start_of_turn>model")[-1].strip()
    if "<end_of_turn>" in result:
        result = result.split("<end_of_turn>")[0].strip()
    return result

suppress = [2028, 3730, 818, 3772, 434, 1582, 16336]

for strength in [1.0, 3.0, 5.0]:
    print(f"\n{'='*80}")
    print(f"PROJECTION SUPPRESSION | strength={strength} | features={suppress}")
    print(f"{'='*80}")
    for label, prompt in test_prompts:
        out = generate_with_projection_suppression(prompt, suppress, strength=strength)
        print(f"\n[{label}] {prompt}")
        print(f"  → {out[:250]}")


PROJECTION SUPPRESSION | strength=1.0 | features=[2028, 3730, 818, 3772, 434, 1582, 16336]

[Speak French] Please say a sentence in French.
  → Bien sûr! Here's a sentence in French:

**"Bonjour, comment allez-vous?"** (Hello, how are you?)

[Speak German] Please say a sentence in German.
  → Okay, here’s a sentence in German:

**"Ich mag es, gerne mit dir zu plaudern."**

This translates to: “I like to chat with you.”

Would you like me to translate something else, or perhaps give you a more complex sentence?

[Speak English] What is the capital of France?
  → The capital of France is **Paris**. 

It’s a common misconception that Paris is the capital of *all* of France, but it’s the capital of the French Republic.

[Speak Spanish] Por favor, dime una oración en español.
  → Claro, aquí tienes una oración:

"El sol brilla intensamente hoy y hace un día perfecto para disfrutar al aire libre." 

¿Te gustaría que te escribiera otra?

[Fr→En translate] Translate to English: 'Le chat dort 

In [12]:
# Finer sweep around the critical region + more test prompts
additional_prompts = [
    ("Write French paragraph", "Write a short paragraph in French about the weather."),
    ("French input→respond", "Répondez en français: Quelle est votre couleur préférée?"),
    ("German paragraph", "Write a short paragraph in German about the weather."),
    ("English reasoning", "Explain briefly why the sky is blue."),
]

all_prompts = test_prompts + additional_prompts

for strength in [2.0, 3.0, 4.0]:
    print(f"\n{'='*80}")
    print(f"STRENGTH = {strength}")
    print(f"{'='*80}")
    for label, prompt in all_prompts:
        out = generate_with_projection_suppression(prompt, suppress, strength=strength, max_new_tokens=80)
        print(f"\n[{label}]")
        print(f"  → {out[:300]}")


STRENGTH = 2.0

[Speak French]
  → Here’s a sentence in French:

**“이것은 정말 멋진 영화입니다.”** (This is a really great movie.)

Would you like me to translate it, or perhaps give you a different sentence?

[Speak German]
  → Okay, here’s a sentence in German:

**"Ich mag es, gerne mit dir zu plaudern."**

This translates to: “I like to chat with you.”

Would you like me to translate something else, or perhaps give you a more complex sentence?

[Speak English]
  → The capital of France is **Paris**. 

It’s a common misconception that Paris is the capital of *all* of France, but it’s the capital of the French Republic.

[Speak Spanish]
  → Claro, aquí tienes una oración:

"El sol brilla intensamente hoy y hace un día perfecto para disfrutar al aire libre." 

¿Te gustaría que te escribiera otra?

[Fr→En translate]
  → The cat is sleeping peacefully on the red rug. 

Or, a slightly more poetic translation:

The cat is sleeping peacefully on the red rug.

[General capability]
  → 4

[Write French

## Results Summary\n\nSuppressing 7 French-specific SAE features (2028, 3730, 818, 3772, 434, 1582, 16336) at layer 22 with projection strength 2-4:\n\n- **French production is selectively disabled** — model substitutes Korean or English\n- **German, Spanish, English completely unaffected**\n- **French comprehension preserved** — translation from French still works\n- **General capabilities intact** — reasoning, math all fine\n- The Korean substitution suggests the model still activates "non-English language output" but with French suppressed, it falls to the next most active language attractor\n\n### Open questions\n1. Is this suppression bidirectional? (Does it affect French *comprehension* at higher strengths?)\n2. Why Korean specifically as the fallback?\n3. How many features are actually needed? (Is feature 2028 alone sufficient?)